In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 6
INTERVAL = "5m"
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "BTCUSDT"
INTERVAL = "5m"
TARGET_HORIZON = 6
MODEL_TYPE = "xgb"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split
from models import tune_selected_features_only , make_bucket_table, fit_final_model

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_{INTERVAL}.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")

features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 83,520


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,vol_regime_ratio,hour_sin,hour_cos,dow_sin,dow_cos,macd,macd_signal,macd_hist,atr_14,atr_norm
0,2025-06-01 00:00:00+00:00,104591.88,104647.11,104530.42,104530.43,44.40977,2025-06-01 00:04:59.999999+00:00,4.644729e+06,8151,14.88668,...,NaN,0.0,1.0,-0.781831,0.62349,0.000000,0.000000,0.000000,NaN,NaN
1,2025-06-01 00:05:00+00:00,104530.43,104559.56,104509.21,104535.84,22.60329,2025-06-01 00:09:59.999999+00:00,2.362841e+06,6240,10.39144,...,NaN,0.0,1.0,-0.781831,0.62349,0.431567,0.086313,0.345254,NaN,NaN
2,2025-06-01 00:10:00+00:00,104535.84,104536.59,104454.41,104473.01,24.19999,2025-06-01 00:14:59.999999+00:00,2.528990e+06,5530,7.69750,...,NaN,0.0,1.0,-0.781831,0.62349,-4.247309,-0.780411,-3.466898,NaN,NaN
3,2025-06-01 00:15:00+00:00,104473.01,104487.81,104396.22,104462.18,42.12392,2025-06-01 00:19:59.999999+00:00,4.399314e+06,11415,17.38966,...,NaN,0.0,1.0,-0.781831,0.62349,-8.728624,-2.370054,-6.358570,NaN,NaN
4,2025-06-01 00:20:00+00:00,104462.17,104490.57,104374.79,104433.71,22.53878,2025-06-01 00:24:59.999999+00:00,2.354018e+06,10547,10.08051,...,NaN,0.0,1.0,-0.781831,0.62349,-14.411265,-4.778296,-9.632969,NaN,NaN


In [8]:
target_col = f"target_{TARGET_HORIZON}"
ret_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col, ret_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col, ret_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]
fwd_ret_train = train_df[ret_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]
fwd_ret_valid = valid_df[ret_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]
fwd_ret_test = test_df[ret_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 83,454
[info] optuna train rows: 53,410
[info] valid rows:        13,353
[info] test rows:         16,691


In [9]:
results = tune_selected_features_only(
    model_type=MODEL_TYPE,
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    fwd_ret_valid=fwd_ret_valid,
    n_trials=100,
    objective_metric="roc_auc",
    top_k=25,
)

print(results["selected_features"])
print(results["feature_importance"].head(30))

[I 2026-03-23 14:29:18,237] A new study created in memory with name: no-name-f35a39f6-321f-4999-b7e4-a092893d6a5c


[I 2026-03-23 14:29:18,440] Trial 0 finished with value: 0.5422111526646576 and parameters: {'n_estimators': 500, 'learning_rate': 0.046187109390049115, 'max_depth': 5, 'subsample': 0.7996646210492592, 'colsample_bytree': 0.6890046601106091, 'colsample_bylevel': 0.6889986300840507, 'min_child_weight': 5, 'gamma': 2.5985284373248057, 'reg_alpha': 0.12306931514988033, 'reg_lambda': 8.341106432362084, 'scale_pos_weight': 0.9055753866472965}. Best is trial 0 with value: 0.5422111526646576.


[I 2026-03-23 14:29:18,708] Trial 1 finished with value: 0.5481117650002929 and parameters: {'n_estimators': 900, 'learning_rate': 0.03818145165896871, 'max_depth': 3, 'subsample': 0.6954562418017751, 'colsample_bytree': 0.6958511274633585, 'colsample_bylevel': 0.7260605607398845, 'min_child_weight': 13, 'gamma': 1.2958350559263474, 'reg_alpha': 0.010295300642650052, 'reg_lambda': 6.252287916406214, 'scale_pos_weight': 0.9462110593183567}. Best is trial 1 with value: 0.5481117650002929.


[I 2026-03-23 14:29:19,055] Trial 2 finished with value: 0.54828929587574 and parameters: {'n_estimators': 500, 'learning_rate': 0.018033330377234345, 'max_depth': 4, 'subsample': 0.8462939903482534, 'colsample_bytree': 0.6999184455395899, 'colsample_bylevel': 0.778558609603403, 'min_child_weight': 14, 'gamma': 0.13935123815999317, 'reg_alpha': 0.12957079329680446, 'reg_lambda': 1.6666983286066417, 'scale_pos_weight': 0.9205630160045514}. Best is trial 2 with value: 0.54828929587574.


[I 2026-03-23 14:29:19,267] Trial 3 finished with value: 0.5494571393110141 and parameters: {'n_estimators': 900, 'learning_rate': 0.047309442068985116, 'max_depth': 5, 'subsample': 0.7261534422933427, 'colsample_bytree': 0.6744180285015959, 'colsample_bylevel': 0.8210582566280392, 'min_child_weight': 12, 'gamma': 0.3661147045343365, 'reg_alpha': 0.05269751777340593, 'reg_lambda': 1.1085122517311703, 'scale_pos_weight': 1.2572038600560156}. Best is trial 3 with value: 0.5494571393110141.


...

[I 2026-03-23 14:30:01,010] Trial 96 finished with value: 0.5520940339077238 and parameters: {'n_estimators': 500, 'learning_rate': 0.013151406021493638, 'max_depth': 3, 'subsample': 0.694063442775472, 'colsample_bytree': 0.7995716911137614, 'colsample_bylevel': 0.801001798809107, 'min_child_weight': 12, 'gamma': 1.919717449686581, 'reg_alpha': 0.001409589288393662, 'reg_lambda': 3.203111624109217, 'scale_pos_weight': 1.1769218691463548}. Best is trial 35 with value: 0.5568464636231156.


[I 2026-03-23 14:30:01,404] Trial 97 finished with value: 0.550948129338537 and parameters: {'n_estimators': 700, 'learning_rate': 0.02036496600048088, 'max_depth': 3, 'subsample': 0.6636359180072151, 'colsample_bytree': 0.7527848302316174, 'colsample_bylevel': 0.8279504083298257, 'min_child_weight': 17, 'gamma': 2.271227220266169, 'reg_alpha': 0.0020725062579926884, 'reg_lambda': 1.668015946021763, 'scale_pos_weight': 1.2014084045942168}. Best is trial 35 with value: 0.5568464636231156.


[I 2026-03-23 14:30:01,796] Trial 98 finished with value: 0.5551328642418478 and parameters: {'n_estimators': 600, 'learning_rate': 0.016287235869570874, 'max_depth': 3, 'subsample': 0.6757312338661151, 'colsample_bytree': 0.7831615710925017, 'colsample_bylevel': 0.7913110052773676, 'min_child_weight': 20, 'gamma': 1.987194066134173, 'reg_alpha': 0.0011667177192516613, 'reg_lambda': 1.0670558079600705, 'scale_pos_weight': 1.2729240785302234}. Best is trial 35 with value: 0.5568464636231156.


[I 2026-03-23 14:30:02,188] Trial 99 finished with value: 0.5508613165160048 and parameters: {'n_estimators': 500, 'learning_rate': 0.018444798133465293, 'max_depth': 3, 'subsample': 0.7676177148151543, 'colsample_bytree': 0.7728515209646949, 'colsample_bylevel': 0.7909798106446057, 'min_child_weight': 20, 'gamma': 2.009860220240113, 'reg_alpha': 0.0011193470208282785, 'reg_lambda': 1.0940133678810244, 'scale_pos_weight': 1.2736490496162718}. Best is trial 35 with value: 0.5568464636231156.


['dow_cos', 'vol_30', 'hour_cos', 'atr_norm', 'hour_sin', 'dow_sin', 'mom_60', 'dist_ma_30', 'mom_15', 'dist_ma_15', 'vol_regime_ratio', 'imbalance_15', 'trend_strength', 'range_ratio', 'macd_hist', 'mom_5', 'vol_5', 'trades_z', 'vol_ratio_5_30', 'bar_range', 'volume_z', 'imbalance', 'num_trades_mom_5', 'imbalance_z', 'co_spread']
feature
dow_cos             9.866817
vol_30              9.846213
hour_cos            9.533494
atr_norm            9.518166
hour_sin            9.413604
dow_sin             9.318570
mom_60              9.222382
dist_ma_30          9.108226
mom_15              9.091309
dist_ma_15          9.033381
vol_regime_ratio    8.885883
imbalance_15        8.634820
trend_strength      8.278977
range_ratio         8.259020
macd_hist           8.132592
mom_5               7.952106
vol_5               7.938831
trades_z            7.515620
vol_ratio_5_30      7.510489
bar_range           7.081801
volume_z            7.074646
imbalance           7.030177
num_trades_mom_5    6

In [10]:
artifacts = fit_final_model(
    model_type=MODEL_TYPE,
    best_params=results["best_params"],
    selected_features=results["selected_features"],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

In [11]:
base_model = artifacts["base_model"]
selected_features = artifacts["selected_features"]

X_train_sel = X_train[selected_features].copy()
X_valid_sel = X_valid[selected_features].copy()
X_train_full_sel = pd.concat([X_train_sel, X_valid_sel], axis=0)

X_test_sel = X_test[selected_features].copy()
y_train_full = pd.concat([y_train, y_valid], axis=0)

train_pred = base_model.predict_proba(X_train_full_sel)[:, 1]
test_pred = base_model.predict_proba(X_test_sel)[:, 1]

In [12]:
train_pred_label = (train_pred >= 0.5).astype(int)
test_pred_label = (test_pred >= 0.5).astype(int)

print("[eval] computing metrics...")

train_ic = spearmanr(train_pred, fwd_ret_train)[0]
test_ic = spearmanr(test_pred, fwd_ret_test)[0]

train_auc = roc_auc_score(y_train_full, train_pred)
test_auc = roc_auc_score(y_test, test_pred)

train_pr_auc = average_precision_score(y_train_full, train_pred)
test_pr_auc = average_precision_score(y_test, test_pred)

train_logloss = log_loss(y_train_full, np.clip(train_pred, 1e-8, 1 - 1e-8))
test_logloss = log_loss(y_test, np.clip(test_pred, 1e-8, 1 - 1e-8))

train_brier = brier_score_loss(y_train_full, train_pred)
test_brier = brier_score_loss(y_test, test_pred)

train_acc = accuracy_score(y_train_full, train_pred_label)
test_acc = accuracy_score(y_test, test_pred_label)

train_precision = precision_score(y_train_full, train_pred_label, zero_division=0)
test_precision = precision_score(y_test, test_pred_label, zero_division=0)

train_recall = recall_score(y_train_full, train_pred_label, zero_division=0)
test_recall = recall_score(y_test, test_pred_label, zero_division=0)

train_f1 = f1_score(y_train_full, train_pred_label, zero_division=0)
test_f1 = f1_score(y_test, test_pred_label, zero_division=0)

print("\n===== RESULTS =====")
print(f"Train IC:        {train_ic:.6f}")
print(f"Test IC:         {test_ic:.6f}")
print(f"Train ROC AUC:   {train_auc:.6f}")
print(f"Test ROC AUC:    {test_auc:.6f}")
print(f"Train PR AUC:    {train_pr_auc:.6f}")
print(f"Test PR AUC:     {test_pr_auc:.6f}")
print(f"Train Log Loss:  {train_logloss:.6f}")
print(f"Test Log Loss:   {test_logloss:.6f}")
print(f"Train Brier:     {train_brier:.6f}")
print(f"Test Brier:      {test_brier:.6f}")
print(f"Train Accuracy:  {train_acc:.6f}")
print(f"Test Accuracy:   {test_acc:.6f}")
print(f"Train Precision: {train_precision:.6f}")
print(f"Test Precision:  {test_precision:.6f}")
print(f"Train Recall:    {train_recall:.6f}")
print(f"Test Recall:     {test_recall:.6f}")
print(f"Train F1:        {train_f1:.6f}")
print(f"Test F1:         {test_f1:.6f}")

[eval] computing metrics...

===== RESULTS =====
Train IC:        0.139403
Test IC:         0.040281
Train ROC AUC:   0.579707
Test ROC AUC:    0.526333
Train PR AUC:    0.575368
Test PR AUC:     0.521067
Train Log Loss:  0.694606
Test Log Loss:   0.701602
Train Brier:     0.250742
Test Brier:      0.254164
Train Accuracy:  0.504186
Test Accuracy:   0.500449
Train Precision: 0.503308
Test Precision:  0.499879
Train Recall:    0.998032
Test Recall:     0.995320
Train F1:        0.669159
Test F1:         0.665517


In [13]:
eval_df = pd.DataFrame({
    "pred": test_pred,
    "y_cls": y_test.values,
    "fwd_ret_test": fwd_ret_test.values,   # continuous realised return
})

eval_df["pred_bin"] = pd.qcut(eval_df["pred"], 10, duplicates="drop")
bucket_stats = eval_df.groupby("pred_bin")["fwd_ret_test"].agg(["mean", "count", "std"])
print(bucket_stats)

                    mean  count       std
pred_bin                                 
(0.433, 0.534] -0.000190   1670  0.003558
(0.534, 0.546] -0.000225   1669  0.003884
(0.546, 0.553] -0.000115   1669  0.004313
(0.553, 0.56]  -0.000199   1669  0.004542
(0.56, 0.566]  -0.000160   1669  0.004364
(0.566, 0.573]  0.000015   1669  0.004623
(0.573, 0.58]  -0.000011   1669  0.004792
(0.58, 0.589]  -0.000122   1669  0.005002
(0.589, 0.602]  0.000005   1669  0.004979
(0.602, 0.717]  0.000294   1669  0.005568


/tmp/ipykernel_1343041/3344132490.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_stats = eval_df.groupby("pred_bin")["fwd_ret_test"].agg(["mean", "count", "std"])


In [14]:
top_decile_threshold = float(np.quantile(test_pred, 0.9))
bottom_decile_threshold = float(np.quantile(test_pred, 0.1))

top_decile_mean_ret = float(eval_df.loc[eval_df["pred"] >= top_decile_threshold, "fwd_ret_test"].mean())
bottom_decile_mean_ret = float(eval_df.loc[eval_df["pred"] <= bottom_decile_threshold, "fwd_ret_test"].mean())
overall_mean_ret = float(eval_df["fwd_ret_test"].mean())

signal_threshold = 0.6
signal_rate = float((eval_df["pred"] >= signal_threshold).mean())
signal_mean_ret = float(eval_df.loc[eval_df["pred"] >= signal_threshold, "fwd_ret_test"].mean())

In [15]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/xgb/BTCUSDT__6_predictions.csv


In [16]:
# save model
joblib.dump(artifacts, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(selected_features, f, indent=2)

# save feature importance
results["feature_importance"].to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(results["study"].best_value),
    "model_params": results["best_params"],
    "n_features": int(len(selected_features)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_ic": float(train_ic),
    "test_ic": float(test_ic),
    "train_auc": float(train_auc),
    "test_auc": float(test_auc),
    "train_pr_auc": float(train_pr_auc),
    "test_pr_auc": float(test_pr_auc),
    "train_logloss": float(train_logloss),
    "test_logloss": float(test_logloss),
    "train_brier": float(train_brier),
    "test_brier": float(test_brier),
    "train_accuracy": float(train_acc),
    "test_accuracy": float(test_acc),
    "train_precision": float(train_precision),
    "test_precision": float(test_precision),
    "train_recall": float(train_recall),
    "test_recall": float(test_recall),
    "train_f1": float(train_f1),
    "test_f1": float(test_f1),
    "test_top_decile_threshold": top_decile_threshold,
    "test_bottom_decile_threshold": bottom_decile_threshold,
    "test_top_decile_mean_fwd_ret": top_decile_mean_ret,
    "test_bottom_decile_mean_fwd_ret": bottom_decile_mean_ret,
    "test_overall_mean_fwd_ret": overall_mean_ret,
    "test_signal_threshold": signal_threshold,
    "test_signal_rate": signal_rate,
    "test_signal_mean_fwd_ret": signal_mean_ret,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat(),
    "train_positive_rate": float(y_train_full.mean()),
    "test_positive_rate": float(y_test.mean()),
    "test_pred_mean": float(np.mean(test_pred)),
    "test_pred_std": float(np.std(test_pred)),
    "test_pred_p10": float(np.quantile(test_pred, 0.10)),
    "test_pred_p50": float(np.quantile(test_pred, 0.50)),
    "test_pred_p90": float(np.quantile(test_pred, 0.90)),
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/xgb/BTCUSDT__h6_model.joblib
[saved] features -> models/xgb/BTCUSDT__h6_feature_cols.json
[saved] feature importance -> models/xgb/BTCUSDT__h6_feature_importance.csv
[saved] metadata -> models/xgb/BTCUSDT__h6_meta.json
